In [5]:
import numpy as np
import pandas as pd
import torch


combined_data_v2 = np.load('combined_data_v2.npy')
new_info_v2 = pd.read_csv('new_info_v2.csv')
yeo_parcels_binary = np.load('yeo_parcels_binary.npy')



In [6]:

# Convert arrays to tensors
task_matrix_shared_tensor = torch.tensor(combined_data_v2[:, yeo_parcels_binary == 1], dtype=torch.float32).cuda()
task_names_tensor = torch.tensor(new_info_v2.taskName.values, dtype=torch.long).cuda()


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
def find_top_combinations_gpu(task_matrix, task_names, num_tasks=4, function='inverse_trace', top_n=1, sample_size=100000):
    """Finds the top N combinations of `num_tasks` using random sampling with GPU acceleration."""
    results = []

    # Generate all task indices
    task_indices = torch.arange(len(task_names)).cuda()

    # Randomly sample combinations
    sampled_combinations = torch.stack([task_indices[torch.randint(0, len(task_indices), (num_tasks,))] for _ in range(sample_size)]).cuda()

    for i, comb in enumerate(sampled_combinations):
        if i % 10000 == 0:
            print(f"Processing sample {i+1}/{sample_size}")

        subset_matrix = task_matrix[comb, :]
        subset_matrix = subset_matrix - subset_matrix.mean(dim=0, keepdim=True)

        varcov = torch.matmul(subset_matrix, subset_matrix.t())
        varcov += 0.01 * torch.eye(varcov.shape[0]).cuda()

        if function == 'inverse_trace':
            inverse_cov = torch.inverse(varcov)
            eigenvalues = torch.linalg.eigvals(inverse_cov)
            function_result = torch.sum(eigenvalues).item()
        elif function == 'trace':
            eigenvalues = torch.linalg.eigvals(varcov)
            function_result = torch.sum(eigenvalues).item()

        results.append((function_result, eigenvalues.cpu().numpy(), comb.cpu().numpy()))

    # Sort results
    if function == 'inverse_trace':
        results.sort(reverse=False, key=lambda x: x[0])
    elif function == 'trace':
        results.sort(reverse=True, key=lambda x: x[0])

    top_combinations = results[:top_n]
    return top_combinations


In [ ]:
# Run the algorithm to find top combinations
top_combinations2 = find_top_combinations_gpu(
    task_matrix_shared_tensor, 
    task_names_tensor, 
    num_tasks=12, 
    function='inverse_trace', 
    top_n=1, 
    sample_size=1000000
)

# Print the top combinations with their corresponding task names
for i, (value, eigenvalues, comb) in enumerate(top_combinations2, start=1):
    combo_task_names = [new_info_v2.taskName.values[index] for index in comb]
    print(f'Combination {i}:', combo_task_names)